In [ ]:
import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Decoder_CLasses.ContinuousVoronoiDecoder import ContinuousVoronoiDecoder
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)


In [ ]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
ConeTaped = TesPartsDir / "ConeTaped.stp"
FreeFormCLosed = TesPartsDir / "FreeFormClosed.stp"
Planar = TesPartsDir / "Planar.stp"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"
FreeFormSurf4 = TesPartsDir / "FreeForm4.stp"
FreeFormBench = TesPartsDir / "FreeFormBench.stp"
FreeSharp1 = TesPartsDir / "FreeSharp1.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
SeatBr = TesPartsDir / "SeatBr.stp"
MouseBot = TesPartsDir / "MouseBot.stp"

shape_path = CircularSurf1

Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=128,
)
domain = Face_Cad.generate_from_file(shape_path)
Face_Cad.print_face_info()
viz = CADDomainVisualizer(Face_Cad)
viz.plot_uv_domain()
viz.show_3d(res_u=200, res_v=200, show_edges=True)


In [ ]:
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

In [ ]:
dec = ContinuousVoronoiDecoder(
    return_xyz=False,
    min_area=1e-3,
    duplicate_merge_sigma = 0.005,
    tau_area=1e-4,
    min_seed_dist=0.01,
    tau_close=0.001,
)

for i in range(1,4):
    N=100
    seeds = random_seeds_min_dist(
        N=N,
        min_dist=0.008,
        seed=i*21,
        device="cpu",
    )

    seeds = seeds.detach().clone().requires_grad_(True)

    out = dec(
        seeds,
        topology_mode="scipy",
        return_xyz=False,
    )

    graph = out["graph"]

    # print("interior vertices:", out["vertices_uv"].shape)
    # print("finite edges:", out["edges"]["edge_index"].shape)
    # print("boundary rays:", out["boundary_rays"].shape)

    # print("graph nodes:", graph["nodes_uv"].shape)
    # print("graph edges:", graph["edge_index"].shape)
    # print("node type:", graph["node_type"])
    # print("boundary nodes:", graph["num_boundary_nodes"])


    loss = graph["nodes_uv"].sum()
    loss.backward()
    print("grad finite:", torch.isfinite(seeds.grad).all())

    # dec.plot_generatedN_graph_debug(seeds,out,Face_Cad,True,True,12,False)
    dec.plot_scipy_vs_generated_graph(seeds,out,Face_Cad,True,True,8,False);

In [ ]:
dec = ContinuousVoronoiDecoder(
    return_xyz=False,
    min_area=1e-3,
    duplicate_merge_sigma = 0.005,
    tau_area=1e-4,
    min_seed_dist=0.01,
    tau_close=0.001,
)

for i in range(1,4):
    N =10
    seeds = random_seeds_min_dist(
        N=N,
        min_dist=0.08,
        seed=i,
        device="cpu",
    )
    seeds = seeds.detach().clone().requires_grad_(True)

    out = dec(
        seeds,
        topology_mode="scipy",
        return_xyz=False,
    )

    graph = out["graph"]

    # print("interior vertices:", out["vertices_uv"].shape)
    # print("finite edges:", out["edges"]["edge_index"].shape)
    # print("boundary rays:", out["boundary_rays"].shape)

    # print("graph nodes:", graph["nodes_uv"].shape)
    # print("graph edges:", graph["edge_index"].shape)
    # print("node type:", graph["node_type"])
    # print("boundary nodes:", graph["num_boundary_nodes"])


    # loss = graph["nodes_uv"].sum()
    # loss.backward()
    # print("grad finite:", torch.isfinite(seeds.grad).all())

    dec.plot_scipy_vs_generated_graph(seeds,out,domain,True,True,8)


In [ ]:
from tests.test_dummy_scipy_training import (
    DummyVoronoiSeedTrainer,
    irregular_test_seeds,
)

initial_seeds = irregular_test_seeds(dtype=torch.float32)
initial_seeds= seeds
trainer = DummyVoronoiSeedTrainer(
    initial_seeds,
    learning_rate=5e-3,
    edge_loss_weight=0,  # Set to 0 for area loss only
)

diagnostics = trainer.fit(steps=2000,plot_first_last=True)

trained_seeds = trainer.seeds_uv.detach()
print(trained_seeds)